# Lakehouse Maintenance — OPTIMIZE + VACUUM

Standalone maintenance notebook for compacting small files and removing stale data
across all Delta tables in a target Lakehouse.

| Operation | What It Does | Cost of Re-Running |
|-----------|-------------|--------------------|
| **OPTIMIZE** | Compacts small Parquet files into larger ones. V-Order is applied automatically in Fabric. | Near-zero on already-optimized tables. |
| **VACUUM** | Deletes files no longer referenced by the Delta log older than the retention threshold. | Near-zero — just scans the log. |

**Schedule:** Match your ingestion cadence. Daily or weekly is typical.

**Schema-aware:** Discovers tables across all schemas in a schema-enabled Lakehouse.

## Configuration

Target workspace and Lakehouse are auto-detected from the attached Lakehouse — override the constants below to point at a different environment. `ZORDER_COLUMNS` opts specific tables into ZORDER clustering on high-cardinality filter/join keys.

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────────

context = notebookutils.runtime.context

# --- Lakehouse Target ---
# Auto-detected from the notebook's attached lakehouse.
# Override these if targeting a different workspace or lakehouse.
WORKSPACE_NAME = context.get("currentWorkspaceName")
LAKEHOUSE_NAME = context.get("defaultLakehouseName")

# --- Table Limit ---
# -1 = process all discovered tables.
# Positive integer = cap for testing (e.g., 5 to validate on a subset).
TABLE_LIMIT = -1

# --- Vacuum Retention (hours) ---
# Delta minimum is 168 hours (7 days). Going below requires disabling
# the safety check — not recommended in production.
VACUUM_RETENTION_HOURS = 168

# --- ZORDER Columns (optional) ---
# Map of table name -> list of columns to ZORDER by.
# Tables not listed here skip ZORDER (OPTIMIZE-only).
# ZORDER co-locates related data in the same files for faster filtered reads.
# Best candidates: high-cardinality columns you frequently filter or join on.
ZORDER_COLUMNS = {
    # "`dbo`.`SalesOrder`": ["CustomerId", "OrderDate"],
    # "`dbo`.`TransactionLine`": ["TransactionId"],
}

# --- Strict Mode ---
# False (default) — print summary and continue even if some tables failed.
#                   Right for scheduled runs: one bad table shouldn't stop
#                   maintenance on the rest of the lakehouse.
# True  — raise RuntimeError after the summary when any table failed. Use in
#         CI/ad-hoc runs where you want the notebook exit code to reflect state.
STRICT = False

# --- Derived Path (do not edit) ---
SOURCE_ROOT = (
    f"abfss://{WORKSPACE_NAME}@onelake.dfs.fabric.microsoft.com"
    f"/{LAKEHOUSE_NAME}.Lakehouse/Tables"
)

print("--- Configuration ---")
print(f"  Workspace:        {WORKSPACE_NAME}")
print(f"  Lakehouse:        {LAKEHOUSE_NAME}")
print(f"  Table limit:      {'ALL' if TABLE_LIMIT == -1 else TABLE_LIMIT}")
print(f"  Vacuum retention: {VACUUM_RETENTION_HOURS} hours")
print(f"  ZORDER targets:   {len(ZORDER_COLUMNS)} table(s)")
print(f"  Strict mode:      {STRICT}")
print(f"  Source root:      {SOURCE_ROOT}")

## Discover Delta Tables

Scans all schemas under the Lakehouse Tables path. Validates each directory
contains a `_delta_log` folder before including it — non-Delta folders are skipped silently.

In [ ]:
from notebookutils import mssparkutils
import py4j
import datetime

all_tables = []

try:
    schemas = mssparkutils.fs.ls(SOURCE_ROOT)

    for schema in schemas:
        if not schema.isDir:
            continue

        schema_name = schema.name
        print(f"  Scanning schema: {schema_name}")

        try:
            tables = mssparkutils.fs.ls(schema.path)
        except Exception as e:
            print(f"  ⚠️  Could not scan schema '{schema_name}': {e}")
            continue

        for t in tables:
            if not t.isDir:
                continue

            # Confirm Delta table by checking for _delta_log directory
            delta_log_path = f"{t.path}/_delta_log"
            try:
                mssparkutils.fs.ls(delta_log_path)
                spark_table_name = f"`{schema_name}`.`{t.name}`"
                all_tables.append(spark_table_name)
            except py4j.protocol.Py4JJavaError:
                # Expected for non-Delta folders — skip silently
                pass
            except Exception as e:
                print(f"  ⚠️  Error checking {schema_name}/{t.name}: {e}")

except Exception as e:
    print(f"❌ FATAL: Could not list source root '{SOURCE_ROOT}': {e}")
    raise

print(f"\nDiscovered {len(all_tables)} Delta tables across all schemas.")

## OPTIMIZE + VACUUM

Runs OPTIMIZE (with optional ZORDER) then VACUUM on each discovered table.
If OPTIMIZE fails for a table, VACUUM is skipped for that table — the table state
may be uncertain and VACUUM could remove files still needed by in-progress operations.

In [ ]:
if TABLE_LIMIT == -1:
    tables_to_process = all_tables
else:
    tables_to_process = all_tables[:TABLE_LIMIT]

print(f"Processing {len(tables_to_process)} tables.\n")

# Canonical Tier-2 result buckets. A table is "succeeded" only when both
# OPTIMIZE and VACUUM complete cleanly; any operation-level failure routes
# the table into `failed`. Per-table metrics (files added/removed, op status)
# live in `per_table` so the detail table in the summary survives.
results = {
    "succeeded": [],  # list[str]: "`schema`.`table`"
    "skipped":   [],  # list[dict]: {"name": str, "reason": str}
    "failed":    [],  # list[dict]: {"name": str, "error": str}
}
per_table = []       # list[dict]: per-table metrics for summary detail

start_time = datetime.datetime.now()

for table_name in tables_to_process:
    print(f"── {table_name} ──")
    detail = {
        "table": table_name,
        "optimize_status": "SKIPPED",
        "vacuum_status": "SKIPPED",
        "files_removed": 0,
        "files_added": 0,
    }

    # --- OPTIMIZE (with optional ZORDER) ---
    try:
        zorder_cols = ZORDER_COLUMNS.get(table_name, [])
        if zorder_cols:
            zorder_clause = f" ZORDER BY ({', '.join(zorder_cols)})"
            print(f"  OPTIMIZE + ZORDER ({', '.join(zorder_cols)})...", end=" ")
        else:
            zorder_clause = ""
            print(f"  OPTIMIZE...", end=" ")

        optimize_result = spark.sql(f"OPTIMIZE {table_name}{zorder_clause}")

        # Extract compaction metrics from the result DataFrame
        metrics_rows = optimize_result.collect()
        if metrics_rows:
            row = metrics_rows[0]
            try:
                metrics = row["metrics"]
                detail["files_removed"] = metrics["numFilesRemoved"]
                detail["files_added"] = metrics["numFilesAdded"]
                print(f"✅ (removed {metrics['numFilesRemoved']} files → {metrics['numFilesAdded']} files)")
            except Exception:
                # Metrics struct shape can vary by runtime version
                print(f"✅ (metrics not parseable — check Spark UI)")
        else:
            print(f"✅ (no rows returned — table may already be optimized)")

        detail["optimize_status"] = "SUCCESS"

    except Exception as e:
        print(f"❌ FAILED: {e}")
        detail["optimize_status"] = "FAILED"
        results["failed"].append({"name": table_name, "error": f"OPTIMIZE: {e}"})
        per_table.append(detail)
        continue  # Skip VACUUM — table state is uncertain

    # --- VACUUM ---
    try:
        print(f"  VACUUM (retention={VACUUM_RETENTION_HOURS}h)...", end=" ")
        spark.sql(f"VACUUM {table_name} RETAIN {VACUUM_RETENTION_HOURS} HOURS")
        print(f"✅")
        detail["vacuum_status"] = "SUCCESS"
        results["succeeded"].append(table_name)

    except Exception as e:
        print(f"❌ FAILED: {e}")
        detail["vacuum_status"] = "FAILED"
        results["failed"].append({"name": table_name, "error": f"VACUUM: {e}"})

    per_table.append(detail)

end_time = datetime.datetime.now()
elapsed = end_time - start_time

## Summary

In [ ]:
total_files_removed = sum(d["files_removed"] for d in per_table)
total_files_added = sum(d["files_added"] for d in per_table)

print("\n" + "─" * 80)
print("  Summary")
print("─" * 80)
print(f"  Succeeded: {len(results['succeeded'])}")
print(f"  Skipped:   {len(results['skipped'])}")
print(f"  Failed:    {len(results['failed'])}")
print(f"  Elapsed:   {elapsed}")
print(f"  Files removed: {total_files_removed}")
print(f"  Files added:   {total_files_added}")

print(f"\n  {'Table':<45} {'OPTIMIZE':<12} {'VACUUM':<12} {'Removed':>9} {'Added':>9}")
print(f"  {'-'*45} {'-'*12} {'-'*12} {'-'*9} {'-'*9}")

for d in per_table:
    print(
        f"  {d['table']:<45} "
        f"{d['optimize_status']:<12} "
        f"{d['vacuum_status']:<12} "
        f"{d['files_removed']:>9} "
        f"{d['files_added']:>9}"
    )

for entry in results["failed"]:
    print(f"    - {entry['name']}: {entry['error']}")
for entry in results["skipped"]:
    print(f"    ~ {entry['name']}: {entry['reason']}")

print("─" * 80)

if STRICT and results["failed"]:
    raise RuntimeError(
        f"STRICT mode: {len(results['failed'])} operation(s) failed across "
        f"{len({e['name'] for e in results['failed']})} table(s). "
        f"See summary above."
    )